In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import optuna
import sklearn
import xgboost
import shap
import matplotlib.pyplot as plt
import joblib
import json

# Load data
df_wr = pd.read_csv("../data/processed/wr_combined_training.csv")
print(df_wr.shape)

# Features and target
ID_COLS = ["player_id", "player_display_name", "season"]
LEAK_COLS = ["target_fp_ppr", "target_games", "games",
             "fumble_recovery_own_pg", "pick", "round"]
X_wr = df_wr.drop(columns=ID_COLS + LEAK_COLS)
y_wr = df_wr["target_fp_ppr"]
strat_wr = df_wr["college_flag"]

# CV folds
sgkf_wr = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_indices_wr = list(sgkf_wr.split(X_wr, strat_wr, groups=df_wr["player_id"]))

for fold, (train_idx, val_idx) in enumerate(cv_indices_wr):
    train_college = strat_wr.iloc[train_idx].sum()
    val_college = strat_wr.iloc[val_idx].sum()
    print(f"Fold {fold+1} | Train: {len(train_idx)} rows ({train_college} college) "
          f"| Val: {len(val_idx)} rows ({val_college} college)")

# ── Optuna Objectives ──

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=50),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "max_features": trial.suggest_float("max_features", 0.2, 0.8),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
    }

    scores = []
    for fold, (train_idx, val_idx) in enumerate(cv_indices_wr):
        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(X_wr.iloc[train_idx], y_wr.iloc[train_idx])
        preds = model.predict(X_wr.iloc[val_idx])
        rmse = np.sqrt(mean_squared_error(y_wr.iloc[val_idx], preds))
        scores.append(rmse)

        trial.report(rmse, fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return -np.mean(scores)

def objective_xgb(trial):
    params = {
        "n_estimators": 1000,
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 10.0, log=True),
    }

    scores = []
    for fold, (train_idx, val_idx) in enumerate(cv_indices_wr):
        model = XGBRegressor(**params, random_state=42, n_jobs=-1, verbosity=0,
                             early_stopping_rounds=50)
        model.fit(X_wr.iloc[train_idx], y_wr.iloc[train_idx],
                  eval_set=[(X_wr.iloc[val_idx], y_wr.iloc[val_idx])],
                  verbose=False)
        preds = model.predict(X_wr.iloc[val_idx])
        rmse = np.sqrt(mean_squared_error(y_wr.iloc[val_idx], preds))
        scores.append(rmse)

        trial.report(rmse, fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return -np.mean(scores)

# ── Run Studies ──

optuna.logging.set_verbosity(optuna.logging.WARNING)
pruner = optuna.pruners.MedianPruner(n_startup_trials=10)

study_rf = optuna.create_study(direction="maximize", study_name="wr_rf", pruner=pruner)
study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

study_xgb = optuna.create_study(direction="maximize", study_name="wr_xgb", pruner=pruner)
study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

# ── Model Comparison ──

rf_rmse = -study_rf.best_value
xgb_rmse = -study_xgb.best_value

print("\n--- wr Model Comparison ---")
print(f"Random Forest RMSE: {rf_rmse:.4f}  (best trial: {study_rf.best_trial.number})")
print(f"XGBoost RMSE:       {xgb_rmse:.4f}  (best trial: {study_xgb.best_trial.number})")

# ── Refit Best Model on Full Data ──

if rf_rmse < xgb_rmse:
    best_model_wr = RandomForestRegressor(
        **study_rf.best_params, random_state=42, n_jobs=-1
    )
    model_name_wr = "rf"
    best_params_wr = study_rf.best_params
else:
    # refit xgb without early stopping for final model
    xgb_params = {k: v for k, v in study_xgb.best_params.items()}
    xgb_params["n_estimators"] = 1000
    best_model_wr = XGBRegressor(
        **xgb_params, random_state=42, n_jobs=-1, verbosity=0
    )
    model_name_wr = "xgb"
    best_params_wr = study_xgb.best_params

best_model_wr.fit(X_wr, y_wr)

# ── SHAP ──

explainer_wr = shap.TreeExplainer(best_model_wr)
shap_values_wr = explainer_wr.shap_values(X_wr)

shap.summary_plot(shap_values_wr, X_wr, show=False)
plt.title("wr SHAP Beeswarm - Direction and Magnitude")
plt.tight_layout()
plt.savefig(f"wr_{model_name_wr}_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()

shap.summary_plot(shap_values_wr, X_wr, plot_type="bar", show=False)
plt.title("wr SHAP Feature Importance (Mean |SHAP|)")
plt.tight_layout()
plt.savefig(f"wr_{model_name_wr}_shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()

# ── Save Artifacts ──

joblib.dump(best_model_wr, f"best_{model_name_wr}_wr_model.joblib")
joblib.dump(study_rf, "wr_study_rf.joblib")
joblib.dump(study_xgb, "wr_study_xgb.joblib")

feature_cols_wr = X_wr.columns.tolist()
with open(f"{model_name_wr}_wr_feature_cols.json", "w") as f:
    json.dump(feature_cols_wr, f)

metadata_wr = {
    "model_name": model_name_wr,
    "rf_rmse": round(rf_rmse, 4),
    "xgb_rmse": round(xgb_rmse, 4),
    "best_params": best_params_wr,
    "n_features": len(feature_cols_wr),
    "n_samples": len(X_wr),
    "college_pct": round(strat_wr.mean(), 3),
    "rf_trials": len(study_rf.trials),
    "xgb_trials": len(study_xgb.trials),
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "optuna_version": optuna.__version__,
}
with open("wr_model_metadata.json", "w") as f:
    json.dump(metadata_wr, f, indent=2)

print(f"\nwr {model_name_wr.upper()} model saved")
print(f"Best params: {best_params_wr}")

(3234, 24)
Fold 1 | Train: 2587 rows (367 college) | Val: 647 rows (92 college)
Fold 2 | Train: 2587 rows (367 college) | Val: 647 rows (92 college)
Fold 3 | Train: 2587 rows (367 college) | Val: 647 rows (92 college)
Fold 4 | Train: 2588 rows (368 college) | Val: 646 rows (91 college)
Fold 5 | Train: 2587 rows (367 college) | Val: 647 rows (92 college)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]


--- wr Model Comparison ---
Random Forest RMSE: 3.4848  (best trial: 0)
XGBoost RMSE:       3.4771  (best trial: 3)

wr XGB model saved
Best params: {'max_depth': 4, 'learning_rate': 0.019980928641117418, 'subsample': 0.7656617644213658, 'colsample_bytree': 0.793310789626942, 'min_child_weight': 15, 'reg_alpha': 0.4263219565061116, 'reg_lambda': 3.6567876872994747}
